# GJ 876: haze chemistry on the runaway points

In the main sweep, the seven hottest GJ 876 points (1079-1400 W/m2) had **no photochemical steady state**: with Photochem's haze-forming reactions removed (`remove_reaction_particles=True`), C4H4 piled up without bound. Here the same coupled climate-photochemistry loop is rerun on those points with the haze reactions **kept** (`haze=True` in `sweeplib.run_point`).

Haze network: `C2H + C4H2 -> HCaer1 + H`, `H2CN + HCN -> HCaer2`, `C4H + HCCCN -> HCaer3`. Particles use Photochem's default radius (1e-4 cm) unless stated.

**Caveat:** haze acts in the photochemistry only (a carbon sink and a settling species). The climate model ignores haze optical effects (no anti-greenhouse, no UV-shielding feedback on temperature).

In [ ]:
import json, glob, csv, os
import numpy as np, matplotlib.pyplot as plt
def load(d):
    return {round(m['S_Wm2'],1): m for m in (json.load(open(p)) for p in glob.glob(f'{d}/GJ876_S*.json'))}
haze = load('sweep_results_haze'); S = sorted(haze)
nohaze = {s: m for s, m in load('sweep_results').items() if s in haze}
diag = {round(float(r['S_Wm2']),1): r for r in csv.DictReader(open('haze_diagnostics.csv'))}
print(f"{'S':>7} {'no haze':<28} {'pCO2':>9} {'T':>6} | {'haze: stop':<36} {'pCO2':>9} {'T':>6} {'CO':>8} {'CH4':>8} {'haze C/in':>10}")
for s in S:
    a,b=nohaze[s],haze[s]
    print(f"{s:7.1f} {a['photochem_stop']:<28} {a['pCO2_bar']:9.2e} {a['T_surf_K']:6.1f} | {b['photochem_stop']:<36} {b['pCO2_bar']:9.2e} {b['T_surf_K']:6.1f} {b['vmr']['CO']:8.1e} {b['vmr']['CH4']:8.1e} {float(diag[s]['haze_carbon_over_reduced_carbon_input']):10.2f}")
print("\nhaze C/in = carbon locked in haze / carbon supplied as CH4+CO+HCN+CH3CN surface fluxes (can exceed 1: CO2, held at fixed pressure, also supplies carbon)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)
c_n, c_h = '#eb6834', '#2a78d6'
for a, key, lab, log in [(ax[0],'pCO2_bar','CO$_2$ partial pressure (bar)',True),(ax[1],'T_surf_K','Surface temperature (K)',False)]:
    a.plot(S,[nohaze[s][key] for s in S],'^--',color=c_n,mfc='none',label='no haze (runaway, no steady state)')
    a.plot(S,[haze[s][key] for s in S],'o-',color=c_h,label='haze (steady)')
    a.set_xlabel('Instellation (W/m$^2$)'); a.set_ylabel(lab)
    if log: a.set_yscale('log')
ax[1].axhline(288,color='gray',lw=.8,ls=':'); ax[0].legend(fontsize=8)
ax[2].plot(S,[float(diag[s]['haze_carbon_atoms_cm-2_s-1']) for s in S],'o-',color=c_h)
ax[2].set_yscale('log'); ax[2].set_xlabel('Instellation (W/m$^2$)'); ax[2].set_ylabel('Carbon into haze (atoms cm$^{-2}$ s$^{-1}$)')
plt.show()

In [ ]:
print('S   ', '  '.join(f'{s:8.1f}' for s in S))
for k in ['CO2','CO','O2','H2O','CH4','H2']:
    print(k.ljust(4), '  '.join(f"{haze[s]['vmr'][k]:.2e}" for s in S))

In [ ]:
# particle-radius sensitivity (1e-5 cm instead of the default 1e-4 cm), if those runs are present
if os.path.isdir('sweep_results_haze_r1e-5'):
    r5 = load('sweep_results_haze_r1e-5')
    for s in sorted(r5):
        for lab, m in [('radius 1e-4 cm', haze[s]), ('radius 1e-5 cm', r5[s])]:
            print(f"S={s} {lab}: pCO2 {m['pCO2_bar']:.3e} bar, T {m['T_surf_K']:.2f} K, CO {m['vmr']['CO']:.2e}, CH4 {m['vmr']['CH4']:.2e}, steady={m['photochem_steady']}")